## Solution of the Previous Lab Exercise
Below is the code to solve the homework exercise from Exercise 3. It fetches the last 200 vulnerabilities related to a specific topic, extracts the days of the week they were published, and plots a bar chart to show the frequency using Pandas and Matplotlib.

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
params = {
    "keywordSearch": "router",
    "resultsPerPage": 200,
    "startIndex": 0
}

print("[*] Fetching data from NVD API...")
response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    vulns = data.get("vulnerabilities", [])
    
    records = []
    for v in vulns:
        cve = v.get("cve", {})
        cve_id = cve.get("id")
        pub_date = cve.get("published")
        if cve_id and pub_date:
            records.append({"cve_id": cve_id, "published": pub_date})
            
    df = pd.DataFrame(records)
    df["published"] = pd.to_datetime(df["published"])
    
    df["day_of_week"] = df["published"].dt.day_name()
    
    day_counts = df["day_of_week"].value_counts()
    
    most_vuln_day = day_counts.idxmax()
    print(f"\n[+] Day with the most vulnerabilities: {most_vuln_day} ({day_counts.max()} vulnerabilities)\n")
    
    print("Vulnerabilities by Day of the Week:")
    for day, count in day_counts.items():
        print(f"{day}: {count}")
        
    plt.figure(figsize=(10, 6))
    day_counts.plot(kind="bar", color="skyblue", edgecolor="black")
    plt.title("Number of Vulnerabilities Detected by Day of the Week")
    plt.xlabel("Day of the Week")
    plt.ylabel("Number of Vulnerabilities")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print(f"[-] Failed to fetch data. Status code: {response.status_code}\n")


# Network Scanning & Observability with Python

In this tutorial, we will explore three different methods to perform network observability (such as ping sweeps and port scanning) directly within a Python environment.

1. **The Subprocess Module**: Hooking into OS-level commands (like `ping` or terminal `nmap`).
2. **The Socket Library**: Developing low-level, native network interactions.
3. **The python-nmap Library**: Orchestrating advanced Nmap scans programmatically.

## 1. Network Observability via Subprocess
The `subprocess` module is part of the Python standard library. It allows you to spawn new processes, connect to their input/output/error pipes, and obtain their return codes. It is the easiest way to script terminal commands like `ping`.

In [ ]:
import subprocess

target = "google.com"

print(f"[*] Sending 2 ping packets to {target}...")

result = subprocess.run(
    ["ping", "-c", "2", target],
    stdout=subprocess.PIPE,   
    stderr=subprocess.PIPE,   
    text=True 
)

if result.returncode == 0:
    print("[+] Host is UP!")
    print("\n--- Output Log ---")
    print(result.stdout)
else:
    print("[-] Host is DOWN or unreachable.")

## 2. Low-Level Observability via Sockets
While `subprocess` leverages external command-line tools, Python's built-in `socket` library allows you to interact with the network natively. You can attempt a TCP connection (the "handshake") against specific ports to determine if services are actively listening, essentially building a native port scanner.

In [ ]:
import socket

target_ip = "127.0.0.1"
ports_to_test = [22, 80, 443, 8080]

print(f"[*] Commencing Socket Port Scan against {target_ip}")

for port in ports_to_test:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    
    s.settimeout(1.0)
    
    status_code = s.connect_ex((target_ip, port))
    
    if status_code == 0:
        print(f"[+] Port {port}: OPEN")
    else:
        print(f"[-] Port {port}: CLOSED or FILTERED (code: {status_code})")
        
    s.close()

## 3. High-Level Operations via `python-nmap`
The `python-nmap` library provides Python classes that orchestrate the `nmap` port scanner. Rather than parsing long subprocess outputs manually, `python-nmap` extracts the information directly from the executable and returns logical, easily queryable Python dictionaries.

*(Make sure to run `pip install python-nmap` in your terminal before running this cell)*

In [ ]:
import nmap

target_network = "127.0.0.1"

print("[*] Initializing nmap.PortScanner()...")
scanner = nmap.PortScanner()

scanner.scan(target_network, '22,80', '-sV')

print("\n[*] Scan complete. Results:\n")

for host in scanner.all_hosts():
    print(f"Host: {host} (State: {scanner[host].state()})")
    
    for proto in scanner[host].all_protocols():
        print(f"  Protocol: {proto}")
        
        ports = scanner[host][proto].keys()
        for port in sorted(ports):
            state = scanner[host][proto][port]['state']
            name = scanner[host][proto][port].get('name', 'unknown')
            version = scanner[host][proto][port].get('version', '')
            
            print(f"    -> Port {port}: {state}\t (Service: {name} {version})")

## 4. Advanced: Subnet Scan and Host Discovery
We can also use `python-nmap` to perform comprehensive sweeps against an entire network subnet (e.g., `192.168.1.0/24`), identifying all live hosts rapidly using specific ping-scan flags.

In [ ]:
import nmap
import subprocess
import re

scanner = nmap.PortScanner()
scanner.scan(hosts="192.168.1.0/24", ports="22,80,443", arguments="-T4")

def get_mac_from_arp(ip):
    try:
        pid = subprocess.Popen(["arp", "-n", ip], stdout=subprocess.PIPE)
        s = pid.communicate()[0].decode("utf-8")
        mac = re.search(r"(([a-f\d]{1,2}\:){5}[a-f\d]{1,2})", s)
        if mac:
            return mac.groups()[0]
    except Exception:
        pass
    return "Unknown"

for host in scanner.all_hosts():
    if scanner[host].state() == "up":
        reason = scanner[host]["status"]["reason"]
        print(f"Host UP: {host} (Reason: {reason})")
        
        mac = scanner[host]["addresses"].get("mac")
        if not mac:
            mac = get_mac_from_arp(host)
            
        vendor = scanner[host]["vendor"].get(mac, "Unknown Vendor")
        print(f"  MAC: {mac} ({vendor})")
        
        for proto in scanner[host].all_protocols():
            print(f"  Protocol: {proto}")
            ports = scanner[host][proto].keys()
            for p in sorted(ports):
                port_state = scanner[host][proto][p]["state"]
                port_name = scanner[host][proto][p].get("name", "unknown")
                print(f"    -> Port {p}: {port_state} ({port_name})")
        print("-" * 30)
